# NYC 311 NLP (2025-2026)

This notebook implements Phase 4 by turning complaint and closure text into reusable NLP features. It uses the cleaned analytic slice from `EDA.ipynb`, labels hidden subcategories within major complaint families, groups repeated resolution-language templates, and exports a local subtype artifact for later analysis work.

Notes:
- Source file: `data/analytics/requests_2025_2026_analytic.parquet`
- `2026` is partial-year data and trend comparisons below use aligned YTD windows
- The first pass favors interpretable subtype rules, then uses phrase mining to inspect residual unmatched text


In [1]:
from pathlib import Path
import sqlite3
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore', message='.*tight_layout.*')


In [2]:
DATA_PATH = Path('data/analytics/requests_2025_2026_analytic.parquet')
SQLITE_PATH = Path('data/311_requests.sqlite')
OUTPUT_PATH = Path('data/analytics/requests_2025_2026_issue_subtypes.parquet')

COLUMNS = [
    'unique_key',
    'created_date',
    'created_year',
    'agency',
    'complaint_type',
    'descriptor',
    'additional_details',
    'borough',
    'community_board',
    'status',
    'resolution_days',
    'resolved_with_valid_date',
]

TARGET_FAMILIES = [
    'Illegal Parking',
    'Noise - Residential',
    'HEAT/HOT WATER',
    'Blocked Driveway',
    'Noise - Street/Sidewalk',
    'UNSANITARY CONDITION',
    'Street Condition',
    'Water System',
    'PLUMBING',
    'Dirty Condition',
    'PAINT/PLASTER',
]

df = pd.read_parquet(DATA_PATH, columns=COLUMNS)
print(f'Loaded {len(df):,} rows from {DATA_PATH}')


Loaded 4,767,760 rows from data/analytics/requests_2025_2026_analytic.parquet


In [3]:
def clean_string(series: pd.Series, extra_missing: tuple[str, ...] = ()) -> pd.Series:
    cleaned = series.astype('string').str.strip()
    missing_values = {'', 'nan', 'None', 'NULL', 'NaN', *extra_missing}
    return cleaned.mask(cleaned.isin(missing_values), pd.NA)


ENCODING_REPLACEMENTS = {
    'â': "'",
    'â': '"',
    'â': '"',
    'â': '-',
    'â': '-',
    'â¦': '...',
}


def normalize_text(series: pd.Series) -> pd.Series:
    normalized = clean_string(series).fillna('')
    for bad, good in ENCODING_REPLACEMENTS.items():
        normalized = normalized.str.replace(bad, good, regex=False)

    normalized = (
        normalized
        .str.lower()
        .str.replace(r'[^a-z0-9/&+'' -]+', ' ', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    return normalized.replace('', pd.NA)


def top_ngrams(series: pd.Series, *, top_n: int = 15, sample_size: int = 40000) -> pd.DataFrame:
    text = series.dropna()
    if text.empty:
        return pd.DataFrame(columns=['ngram', 'count'])

    if len(text) > sample_size:
        text = text.sample(sample_size, random_state=42)

    vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=5, stop_words='english')
    try:
        matrix = vectorizer.fit_transform(text)
    except ValueError:
        return pd.DataFrame(columns=['ngram', 'count'])

    counts = np.asarray(matrix.sum(axis=0)).ravel()
    result = pd.DataFrame({
        'ngram': vectorizer.get_feature_names_out(),
        'count': counts,
    })
    return result.sort_values('count', ascending=False).head(top_n).reset_index(drop=True)


text_columns = ['complaint_type', 'descriptor', 'additional_details', 'borough', 'community_board', 'agency', 'status']
for column in text_columns:
    df[column] = clean_string(df[column])

df['complaint_type'] = df['complaint_type'].fillna('Unknown')
df['descriptor'] = df['descriptor'].fillna('Unknown')
df['agency'] = df['agency'].fillna('UNKNOWN')
df['status'] = df['status'].fillna('Unknown')
df['borough'] = clean_string(df['borough'], extra_missing=('Unspecified',)).str.upper()
df['created_date'] = pd.to_datetime(df['created_date'], errors='coerce')

df['descriptor_norm'] = normalize_text(df['descriptor'])
df['details_norm'] = normalize_text(df['additional_details'])
df['text_clean'] = (df['descriptor_norm'].fillna('') + ' | ' + df['details_norm'].fillna('')).str.strip(' |').replace('', pd.NA)

coverage_summary = (
    df.groupby('complaint_type')
    .agg(
        complaints=('unique_key', 'size'),
        descriptor_rate=('descriptor_norm', lambda s: s.notna().mean()),
        details_rate=('details_norm', lambda s: s.notna().mean()),
        resolution_rate=('resolved_with_valid_date', 'mean'),
    )
    .sort_values('complaints', ascending=False)
    .assign(
        descriptor_rate=lambda x: x['descriptor_rate'].map('{:.1%}'.format),
        details_rate=lambda x: x['details_rate'].map('{:.1%}'.format),
        resolution_rate=lambda x: x['resolution_rate'].map('{:.1%}'.format),
        modeled=lambda x: x.index.isin(TARGET_FAMILIES),
    )
)
coverage_summary.head(15)


,complaints,descriptor_rate,details_rate,resolution_rate,modeled
complaint_type,,,,,
Illegal Parking,741702,100.0%,4.1%,100.0%,True
Noise - Residential,559609,100.0%,0.0%,100.0%,True
HEAT/HOT WATER,492195,100.0%,100.0%,97.4%,True
Blocked Driveway,230444,100.0%,0.0%,100.0%,True
Noise - Street/Sidewalk,195248,100.0%,0.0%,100.0%,True
UNSANITARY CONDITION,151723,100.0%,70.9%,92.1%,True
Street Condition,116583,100.0%,0.0%,90.0%,True
Water System,98888,100.0%,100.0%,96.3%,True
PLUMBING,96790,100.0%,100.0%,91.1%,True


In [4]:
def extract_cluster_terms(model: KMeans, vectorizer: TfidfVectorizer, top_n: int = 5) -> dict[int, str]:
    terms = np.asarray(vectorizer.get_feature_names_out())
    cluster_terms = {}
    for cluster_id, center in enumerate(model.cluster_centers_):
        top_idx = center.argsort()[::-1][:top_n]
        cluster_terms[cluster_id] = ', '.join(terms[top_idx])
    return cluster_terms


def build_residual_clusters(
    frame: pd.DataFrame,
    *,
    min_rows: int = 150,
    max_clusters: int = 6,
    max_features: int = 1500,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    assignments = []
    profiles = []

    residual_base = frame.loc[
        frame['issue_subtype'].eq('other_unclassified') & frame['text_clean'].notna(),
        ['complaint_type', 'text_clean', 'descriptor', 'additional_details'],
    ]

    for family, family_frame in residual_base.groupby('complaint_type'):
        if len(family_frame) < min_rows:
            continue

        unique_text_count = family_frame['text_clean'].nunique()
        if unique_text_count < 2:
            continue

        requested_clusters = min(max_clusters, unique_text_count, max(2, len(family_frame) // 1000 + 1))
        if requested_clusters >= len(family_frame):
            requested_clusters = max(2, len(family_frame) - 1)
        if requested_clusters < 2:
            continue

        vectorizer = TfidfVectorizer(
            stop_words='english',
            ngram_range=(1, 2),
            min_df=5,
            max_features=max_features,
        )

        try:
            matrix = vectorizer.fit_transform(family_frame['text_clean'])
        except ValueError:
            continue

        if matrix.shape[1] < 2:
            continue

        model = KMeans(n_clusters=requested_clusters, n_init=10, random_state=42)
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=ConvergenceWarning)
            labels = model.fit_predict(matrix)
        cluster_terms = extract_cluster_terms(model, vectorizer)

        family_assignments = pd.DataFrame({
            'row_index': family_frame.index,
            'complaint_type': family,
            'residual_cluster': labels,
        })
        family_assignments['residual_cluster_label'] = family_assignments['residual_cluster'].map(cluster_terms)
        assignments.append(family_assignments)

        family_examples = (
            family_frame.assign(residual_cluster=labels)
            .groupby('residual_cluster')['text_clean']
            .first()
        )
        family_counts = pd.Series(labels).value_counts().sort_index()
        family_profiles = pd.DataFrame({
            'complaint_type': family,
            'residual_cluster': family_counts.index,
            'cluster_rows': family_counts.values,
        })
        family_profiles['cluster_share'] = family_profiles['cluster_rows'] / len(family_frame)
        family_profiles['cluster_terms'] = family_profiles['residual_cluster'].map(cluster_terms)
        family_profiles['example_text'] = family_profiles['residual_cluster'].map(family_examples)
        profiles.append(family_profiles.sort_values('cluster_rows', ascending=False))

    assignment_frame = pd.concat(assignments, ignore_index=True) if assignments else pd.DataFrame(columns=['row_index', 'complaint_type', 'residual_cluster', 'residual_cluster_label'])
    profile_frame = pd.concat(profiles, ignore_index=True) if profiles else pd.DataFrame(columns=['complaint_type', 'residual_cluster', 'cluster_rows', 'cluster_share', 'cluster_terms', 'example_text'])
    return assignment_frame, profile_frame


RESOLUTION_RULES = [
    {'outcome': 'duplicate_report', 'pattern': r'duplicate of a building-wide condition|duplicate of', 'confidence': 'high'},
    {'outcome': 'unable_to_access_or_inspect', 'pattern': r'unable to gain access|unable to complete the inspection|not able to gain access', 'confidence': 'high'},
    {'outcome': 'summons_issued', 'pattern': r'issued a summons|police issued a summons', 'confidence': 'high'},
    {'outcome': 'violation_issued', 'pattern': r'violations were issued|violation descriptions', 'confidence': 'high'},
    {'outcome': 'condition_corrected', 'pattern': r'condition was corrected|conditions were corrected|took action to fix the condition|corrected without the need to issue a summons', 'confidence': 'high'},
    {'outcome': 'no_evidence_observed', 'pattern': r'observed no evidence of the violation|observed no criminal violation|observed no evidence', 'confidence': 'high'},
    {'outcome': 'no_action_needed', 'pattern': r'police action was not necessary|action was not necessary', 'confidence': 'high'},
    {'outcome': 'no_violation_found', 'pattern': r'did not violate the housing laws|no criminal violation existed|no violation existed', 'confidence': 'high'},
    {'outcome': 'external_status_or_referral', 'pattern': r'website|learn more|status for this request is available|please call| email ', 'confidence': 'medium'},
    {'outcome': 'inspection_completed', 'pattern': r'conducted an inspection|inspected this condition|inspection results', 'confidence': 'medium'},
]


def classify_resolution_outcomes(text: pd.Series) -> pd.DataFrame:
    outcome = pd.Series('other_resolution_language', index=text.index, dtype='string')
    confidence = pd.Series('low', index=text.index, dtype='string')

    for rule in RESOLUTION_RULES:
        match = text.str.contains(rule['pattern'], regex=True, na=False) & outcome.eq('other_resolution_language')
        outcome.loc[match] = rule['outcome']
        confidence.loc[match] = rule['confidence']

    return pd.DataFrame({
        'resolution_outcome_group': outcome,
        'resolution_outcome_confidence': confidence,
        'resolution_text_length': text.str.len().astype('Int64'),
    })


def load_resolution_language_features(
    sqlite_path: Path,
    target_families: list[str],
    *,
    chunk_size: int = 200_000,
) -> pd.DataFrame:
    placeholders = ', '.join('?' for _ in target_families)
    query = f'''
        SELECT unique_key, resolution_description
        FROM requests
        WHERE created_date >= '2025-01-01 00:00:00'
          AND problem_formerly_complaint_type IN ({placeholders})
          AND resolution_description IS NOT NULL
          AND TRIM(resolution_description) <> ''
    '''

    frames = []
    with sqlite3.connect(sqlite_path) as connection:
        for chunk in pd.read_sql_query(query, connection, params=tuple(target_families), chunksize=chunk_size):
            chunk['resolution_description'] = clean_string(chunk['resolution_description'])
            resolution_text = normalize_text(chunk['resolution_description'])
            classified = classify_resolution_outcomes(resolution_text.fillna(''))
            classified.insert(0, 'unique_key', chunk['unique_key'].astype('string'))
            frames.append(classified)

    if not frames:
        return pd.DataFrame(columns=['unique_key', 'resolution_outcome_group', 'resolution_outcome_confidence', 'resolution_text_length'])

    return pd.concat(frames, ignore_index=True)


## Rule-Based Subtype Taxonomy
The first pass uses complaint-family-specific rules built from dominant descriptors and additional-detail phrases. Exact descriptor and detail matches get high confidence, while broader combined-text patterns are marked medium confidence.


In [5]:
SUBTYPE_RULES = {
    'Illegal Parking': [
        {'subtype': 'blocked_hydrant', 'pattern': r'blocked hydrant|front of fire hydrant|fire hydrant', 'columns': ['descriptor_norm', 'details_norm'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'posted_sign_violation', 'pattern': r'posted parking sign violation|no standing|no stopping|no parking', 'columns': ['descriptor_norm', 'details_norm'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'blocked_sidewalk', 'pattern': r'blocked sidewalk|parked on sidewalk', 'columns': ['descriptor_norm', 'details_norm'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'commercial_overnight', 'pattern': r'commercial overnight parking', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'overnight_commercial_storage', 'pattern': r'overnight commercial storage|commercial storage', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'detached_trailer', 'pattern': r'detached trailer', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'blocked_crosswalk', 'pattern': r'blocked crosswalk|parked in crosswalk', 'columns': ['descriptor_norm', 'details_norm'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'double_parked', 'pattern': r'double parked blocking traffic|double parked blocking vehicle|double parked', 'columns': ['descriptor_norm', 'details_norm'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'permit_misuse', 'pattern': r'parking permit improper use', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'bike_lane_blocking', 'pattern': r'blocked bike lane', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'plate_issue', 'pattern': r'license plate obscured|paper license plates', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'bus_stop_or_layover', 'pattern': r'unauthorized bus layover|parked at bus stop', 'columns': ['descriptor_norm', 'details_norm'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
    ],
    'Noise - Residential': [
        {'subtype': 'music_party', 'pattern': r'loud music/party|party', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'banging_pounding', 'pattern': r'banging/pounding', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'loud_talking', 'pattern': r'loud talking', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'television', 'pattern': r'loud television', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
    ],
    'HEAT/HOT WATER': [
        {'subtype': 'no_heat', 'pattern': r'^no heat$', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'no_heat_no_hot_water', 'pattern': r'no heat and no hot water', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'no_hot_water', 'pattern': r'^no hot water$', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'heat_on_in_summer', 'pattern': r'heat on in summer', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
    ],
    'Blocked Driveway': [
        {'subtype': 'no_access', 'pattern': r'^no access$', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'partial_access', 'pattern': r'^partial access$', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
    ],
    'Noise - Street/Sidewalk': [
        {'subtype': 'music_party', 'pattern': r'loud music/party|party', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'loud_talking', 'pattern': r'loud talking', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
    ],
    'UNSANITARY CONDITION': [
        {'subtype': 'pests', 'pattern': r'^pests$|roaches|mice|rats|vermin|rodent', 'columns': ['descriptor_norm', 'details_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'mold', 'pattern': r'^mold$|mold|fungus', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'garbage_storage', 'pattern': r'garbage/recycling storage|garbage|recycling', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'sewage', 'pattern': r'^sewage$|sewage', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
    ],
    'Street Condition': [
        {'subtype': 'pothole', 'pattern': r'^pothole$|pothole', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'cave_in', 'pattern': r'cave-in|cave in', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'defective_hardware', 'pattern': r'defective hardware', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'construction_blockage', 'pattern': r'blocked - construction|construction waste|after repaving', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'failed_repair', 'pattern': r'failed street repair|wear & tear|wear tear', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'medium', 'source': 'combined_text_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'rough_or_noisy_road', 'pattern': r'rough pitted or cracked roads|plate condition - noisy|plate condition - shifted', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'medium', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'faded_markings', 'pattern': r'line/marking - faded|marking faded|line marking', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'crash_cushion_defect', 'pattern': r'crash cushion defect|crash cushion|cushion defect', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'guard_rail_issue', 'pattern': r'guard rail - street|guard rail', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'unsafe_worksite', 'pattern': r'unsafe worksite', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'surface_depression', 'pattern': r'depression maintenance|hummock', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'medium', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'plate_condition_open_or_skid', 'pattern': r'plate condition - open|plate condition - anti-skid', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'defacement', 'pattern': r'^defacement$', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'strip_paving', 'pattern': r'^strip paving$', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'maintenance_cover', 'pattern': r'^maintenance cover$', 'columns': ['descriptor_norm'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
    ],
    'Water System': [
        {'subtype': 'hydrant_running', 'pattern': r'hydrant running|hydrant leaking|hydrant defective|hydrant knocked over|fire hydrant emergency', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'dirty_water', 'pattern': r'dirty water', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'system_leak', 'pattern': r'leak|water main break|basement', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'medium', 'source': 'combined_text_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'no_water', 'pattern': r'^no water$|no water', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'low_pressure', 'pattern': r'low water pressure', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'valve_box_cover_missing', 'pattern': r'hyd valve box cover missing|wv2', 'columns': ['descriptor_norm', 'details_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'hydrant_side_post_installation', 'pattern': r'installation of hydrant side post|whfp', 'columns': ['descriptor_norm', 'details_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_or_detail_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'high_pressure', 'pattern': r'high water pressure', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
    ],
    'PLUMBING': [
        {'subtype': 'no_water', 'pattern': r'^no water$', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'faucet_issue', 'pattern': r'faucet broken/missing/leaking', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'broken_missing_fixture', 'pattern': r'broken or missing', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'drain_blocked', 'pattern': r'drain pipe blocked or broken|bowl stopped up', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'no_hot_water', 'pattern': r'^no hot water$|scalding hot water', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'basin_or_pipe_leak', 'pattern': r'basin broken/leaking|pipe leaking|cracked or leaking', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'low_pressure', 'pattern': r'low water pressure', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'chipped_or_rusted_fixture', 'pattern': r'chipped or rusted|chipped rusted', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'sink_detached_from_wall', 'pattern': r'sink detached from wall|detached from wall', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'toilet_runs_continuously', 'pattern': r'continuously runs|toilet continuously runs', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'leak_into_other_apartment', 'pattern': r'leaking into other apartment', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'steam_pipe_or_riser', 'pattern': r'steam pipe/riser|pipe riser|steam pipe|riser', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'medium', 'source': 'combined_text_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'loose_or_wobbly_toilet', 'pattern': r'bowl loose or wobbly|loose or wobbly', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'no_cold_water', 'pattern': r'no cold water', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'radiator_valve_issue', 'pattern': r'shut off valve broken|loose or disconnected', 'columns': ['details_norm', 'combined_text'], 'confidence': 'medium', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'toilet_disconnected', 'pattern': r'^disconnected$', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'basin_missing', 'pattern': r'^basin missing$', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'radiator_missing_or_removed', 'pattern': r'radiator \| missing or removed|radiator missing or removed', 'columns': ['combined_text'], 'confidence': 'high', 'source': 'combined_text_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'toilet_removed_or_missing', 'pattern': r'bowl removed or missing', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'boiler_broken', 'pattern': r'boiler \| broken|boiler broken', 'columns': ['combined_text'], 'confidence': 'high', 'source': 'combined_text_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'sewer_blocked_or_broken', 'pattern': r'broken or blocked pipe|broken or blocked drain', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
    ],
    'Dirty Condition': [
        {'subtype': 'littering', 'pattern': r'littering', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'owner_not_cleaned', 'pattern': r'not cleaned by property owner', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'dog_waste', 'pattern': r'not picked up by dog owner', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'dirt_or_gravel', 'pattern': r'dirt or gravel|dirt gravel', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'broken_glass', 'pattern': r'broken glass', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'syringes', 'pattern': r'^syringes$|syringes', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
        {'subtype': 'car_accident_debris', 'pattern': r'debris from car accident|car accident', 'columns': ['descriptor_norm', 'combined_text'], 'confidence': 'high', 'source': 'descriptor_rule', 'match_column': 'descriptor_norm'},
    ],
    'PAINT/PLASTER': [
        {'subtype': 'chipped_flaking', 'pattern': r'chipped/peeling/flaking|peeling or flaking paint', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'bulging_cracked', 'pattern': r'bulging/hole/cracked|hole or cracked', 'columns': ['details_norm', 'combined_text'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'collapse_risk', 'pattern': r'collapsing or falling', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'dirty_unsanitary', 'pattern': r'dirty or unsanitary', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
        {'subtype': 'tile_broken_missing', 'pattern': r'tile broken or missing', 'columns': ['details_norm'], 'confidence': 'high', 'source': 'additional_details_rule', 'match_column': 'details_norm'},
    ],
}


def assign_subtypes(frame: pd.DataFrame, rules: dict[str, list[dict[str, object]]]) -> pd.DataFrame:
    enriched = frame.copy()
    modeled = enriched['complaint_type'].isin(rules)

    enriched['issue_family'] = enriched['complaint_type']
    enriched['subtype_modeled_flag'] = modeled
    enriched['issue_subtype'] = np.where(modeled, 'other_unclassified', 'not_modeled')
    enriched['issue_subtype_confidence'] = np.where(modeled, 'low', 'not_modeled')
    enriched['subtype_source'] = np.where(modeled, 'unmatched', 'not_modeled')
    enriched['subtype_match_text'] = pd.NA

    for family, family_rules in rules.items():
        idx = enriched.index[enriched['complaint_type'].eq(family)]
        if len(idx) == 0:
            continue

        descriptor = enriched.loc[idx, 'descriptor_norm'].fillna('')
        details = enriched.loc[idx, 'details_norm'].fillna('')
        combined = (descriptor + ' | ' + details).str.strip(' |')
        unresolved = enriched.loc[idx, 'issue_subtype'].eq('other_unclassified').to_numpy(copy=True)

        for rule in family_rules:
            match = np.zeros(len(idx), dtype=bool)
            if 'descriptor_norm' in rule['columns']:
                match |= descriptor.str.contains(rule['pattern'], regex=True, na=False).to_numpy()
            if 'details_norm' in rule['columns']:
                match |= details.str.contains(rule['pattern'], regex=True, na=False).to_numpy()
            if 'combined_text' in rule['columns']:
                match |= combined.str.contains(rule['pattern'], regex=True, na=False).to_numpy()

            assign = match & unresolved
            if not assign.any():
                continue

            assign_idx = idx[assign]
            enriched.loc[assign_idx, 'issue_subtype'] = rule['subtype']
            enriched.loc[assign_idx, 'issue_subtype_confidence'] = rule['confidence']
            enriched.loc[assign_idx, 'subtype_source'] = rule['source']

            if rule['match_column'] == 'descriptor_norm':
                enriched.loc[assign_idx, 'subtype_match_text'] = enriched.loc[assign_idx, 'descriptor']
            elif rule['match_column'] == 'details_norm':
                enriched.loc[assign_idx, 'subtype_match_text'] = enriched.loc[assign_idx, 'additional_details']
            else:
                enriched.loc[assign_idx, 'subtype_match_text'] = combined.loc[assign]

            unresolved[assign] = False

    return enriched


MISMATCH_RULES = [
    {
        'complaint_type': 'Illegal Parking',
        'suspected_family': 'Noise',
        'pattern': r'loud music/party|banging/pounding|loud talking|loud television',
        'columns': ['descriptor_norm', 'combined_text'],
    },
]


def flag_label_mismatches(frame: pd.DataFrame, rules: list[dict[str, object]]) -> pd.DataFrame:
    enriched = frame.copy()
    enriched['potential_label_mismatch_flag'] = False
    enriched['mismatch_suspected_family'] = pd.Series(pd.NA, index=enriched.index, dtype='string')
    enriched['mismatch_match_text'] = pd.Series(pd.NA, index=enriched.index, dtype='string')

    unresolved_mask = enriched['issue_subtype'].eq('other_unclassified')

    for rule in rules:
        family_idx = enriched.index[unresolved_mask & enriched['complaint_type'].eq(rule['complaint_type'])]
        if len(family_idx) == 0:
            continue

        descriptor = enriched.loc[family_idx, 'descriptor_norm'].fillna('')
        details = enriched.loc[family_idx, 'details_norm'].fillna('')
        combined = (descriptor + ' | ' + details).str.strip(' |')

        match = np.zeros(len(family_idx), dtype=bool)
        if 'descriptor_norm' in rule['columns']:
            match |= descriptor.str.contains(rule['pattern'], regex=True, na=False).to_numpy()
        if 'details_norm' in rule['columns']:
            match |= details.str.contains(rule['pattern'], regex=True, na=False).to_numpy()
        if 'combined_text' in rule['columns']:
            match |= combined.str.contains(rule['pattern'], regex=True, na=False).to_numpy()

        if not match.any():
            continue

        match_idx = family_idx[match]
        enriched.loc[match_idx, 'potential_label_mismatch_flag'] = True
        enriched.loc[match_idx, 'mismatch_suspected_family'] = rule['suspected_family']
        enriched.loc[match_idx, 'mismatch_match_text'] = enriched.loc[match_idx, 'descriptor'].fillna(enriched.loc[match_idx, 'additional_details'])

    return enriched


nlp_df = assign_subtypes(df, SUBTYPE_RULES)
nlp_df = flag_label_mismatches(nlp_df, MISMATCH_RULES)
modeled_rows = int(nlp_df['subtype_modeled_flag'].sum())
assigned_rows = int((nlp_df['subtype_modeled_flag'] & nlp_df['issue_subtype'].ne('other_unclassified')).sum())
mismatch_rows = int(nlp_df['potential_label_mismatch_flag'].sum())
print(f'Modeled families cover {modeled_rows:,} complaints ({modeled_rows / len(nlp_df):.1%} of the scoped dataset).')
print(f'Assigned subtypes cover {assigned_rows:,} modeled complaints ({assigned_rows / modeled_rows:.1%} within modeled families).')


Modeled families cover 2,842,077 complaints (59.6% of the scoped dataset).
Assigned subtypes cover 2,842,076 modeled complaints (100.0% within modeled families).


In [6]:
subtype_summary = (
    nlp_df.loc[nlp_df['subtype_modeled_flag']]
    .groupby('complaint_type')
    .agg(
        complaints=('unique_key', 'size'),
        subtype_variants=('issue_subtype', 'nunique'),
        assigned_share=('issue_subtype', lambda s: s.ne('other_unclassified').mean()),
        residual_share=('issue_subtype', lambda s: s.eq('other_unclassified').mean()),
    )
    .sort_values('complaints', ascending=False)
    .assign(
        assigned_share=lambda x: x['assigned_share'].map('{:.1%}'.format),
        residual_share=lambda x: x['residual_share'].map('{:.1%}'.format),
    )
)
subtype_summary

top_subtypes = (
    nlp_df.loc[nlp_df['issue_subtype'].isin(['not_modeled', 'other_unclassified']) == False]
    .groupby(['complaint_type', 'issue_subtype'])
    .size()
    .rename('complaints')
    .reset_index()
    .sort_values(['complaint_type', 'complaints'], ascending=[True, False])
)
top_subtypes.groupby('complaint_type').head(6)


,complaint_type,issue_subtype,complaints
0,Blocked Driveway,no_access,171170
1,Blocked Driveway,partial_access,59274
6,Dirty Condition,littering,43629
7,Dirty Condition,owner_not_cleaned,29497
5,Dirty Condition,dog_waste,2264
4,Dirty Condition,dirt_or_gravel,1461
2,Dirty Condition,broken_glass,1392
8,Dirty Condition,syringes,1016
10,HEAT/HOT WATER,no_heat,294097
11,HEAT/HOT WATER,no_heat_no_hot_water,132789


In [7]:
descriptor_preview = (
    nlp_df.loc[nlp_df['complaint_type'].isin(TARGET_FAMILIES)]
    .groupby(['complaint_type', 'descriptor'])
    .size()
    .rename('complaints')
    .reset_index()
    .sort_values(['complaint_type', 'complaints'], ascending=[True, False])
)
descriptor_preview.groupby('complaint_type').head(8)

residual_frames = []
for family in ['Illegal Parking', 'Street Condition', 'Water System', 'UNSANITARY CONDITION']:
    grams = top_ngrams(
        nlp_df.loc[
            nlp_df['complaint_type'].eq(family) & nlp_df['issue_subtype'].eq('other_unclassified'),
            'text_clean',
        ]
    )
    if grams.empty:
        continue
    grams.insert(0, 'complaint_type', family)
    residual_frames.append(grams)

residual_ngrams = pd.concat(residual_frames, ignore_index=True) if residual_frames else pd.DataFrame(columns=['complaint_type', 'ngram', 'count'])
residual_ngrams


,complaint_type,ngram,count


## Model-Assisted Residual Discovery
The rule-based taxonomy covers most modeled complaints. For the remaining `other_unclassified` rows, this section uses TF-IDF plus K-means within each complaint family to surface recurring residual themes without replacing the main interpretable labels.


In [8]:
nlp_df['residual_cluster'] = pd.Series(pd.NA, index=nlp_df.index, dtype='Int64')
nlp_df['residual_cluster_label'] = pd.Series(pd.NA, index=nlp_df.index, dtype='string')

residual_cluster_assignments, residual_cluster_profiles = build_residual_clusters(nlp_df)
if not residual_cluster_assignments.empty:
    nlp_df.loc[residual_cluster_assignments['row_index'], 'residual_cluster'] = residual_cluster_assignments['residual_cluster'].to_numpy()
    nlp_df.loc[residual_cluster_assignments['row_index'], 'residual_cluster_label'] = residual_cluster_assignments['residual_cluster_label'].to_numpy()

residual_cluster_profiles['cluster_share'] = residual_cluster_profiles['cluster_share'].map('{:.1%}'.format)
residual_cluster_profiles.sort_values(['complaint_type', 'cluster_rows'], ascending=[True, False]).head(20)


,complaint_type,residual_cluster,cluster_rows,cluster_share,cluster_terms,example_text


## Resolution Language NLP
Phase 4 should also look at how agencies describe complaint closure. This section derives compact closure-language features from `resolution_description` in the local SQLite store, then groups repeated templates into interpretable outcome buckets.


In [9]:
common_subtypes = (
    nlp_df.loc[nlp_df['issue_subtype'].isin(['not_modeled', 'other_unclassified']) == False, 'issue_subtype']
    .value_counts()
    .loc[lambda s: s >= 1000]
    .index
)

resolution_features = load_resolution_language_features(SQLITE_PATH, TARGET_FAMILIES)
nlp_df = nlp_df.merge(resolution_features, how='left', on='unique_key')
nlp_df['resolution_has_text'] = nlp_df['resolution_outcome_group'].notna()
nlp_df['resolution_outcome_group'] = nlp_df['resolution_outcome_group'].fillna('no_resolution_text')
nlp_df['resolution_outcome_confidence'] = nlp_df['resolution_outcome_confidence'].fillna('not_applicable')

resolution_outcome_summary = (
    nlp_df.loc[nlp_df['complaint_type'].isin(TARGET_FAMILIES)]
    .groupby(['complaint_type', 'resolution_outcome_group'])
    .size()
    .rename('complaints')
    .reset_index()
    .sort_values(['complaint_type', 'complaints'], ascending=[True, False])
)
resolution_outcome_summary.groupby('complaint_type').head(5)

resolution_by_subtype = (
    nlp_df.loc[
        nlp_df['issue_subtype'].isin(common_subtypes)
        & nlp_df['resolution_outcome_group'].ne('no_resolution_text'),
        ['complaint_type', 'issue_subtype', 'resolution_outcome_group'],
    ]
    .value_counts()
    .rename('complaints')
    .reset_index()
    .sort_values(['complaint_type', 'issue_subtype', 'complaints'], ascending=[True, True, False])
)
resolution_by_subtype.groupby(['complaint_type', 'issue_subtype']).head(3).head(24)


,complaint_type,issue_subtype,resolution_outcome_group,complaints
7,Blocked Driveway,no_access,summons_issued,57894
16,Blocked Driveway,no_access,no_evidence_observed,43630
21,Blocked Driveway,no_access,other_resolution_language,37702
42,Blocked Driveway,partial_access,summons_issued,18602
62,Blocked Driveway,partial_access,other_resolution_language,12500
63,Blocked Driveway,partial_access,no_evidence_observed,12450
178,Dirty Condition,broken_glass,other_resolution_language,1350
285,Dirty Condition,broken_glass,external_status_or_referral,31
174,Dirty Condition,dirt_or_gravel,other_resolution_language,1396
277,Dirty Condition,dirt_or_gravel,external_status_or_referral,45


## Operations and Geography Tie-Ins
These views turn the Phase 4 output into reusable analysis. The point is not only to label complaint text, but also to see which hidden issue groups are concentrated, slower to resolve, or shifting over time.


In [10]:
common_subtypes = (
    nlp_df.loc[nlp_df['issue_subtype'].isin(['not_modeled', 'other_unclassified']) == False, 'issue_subtype']
    .value_counts()
    .loc[lambda s: s >= 1000]
    .index
)

resolution_summary = (
    nlp_df.loc[
        nlp_df['resolved_with_valid_date']
        & nlp_df['issue_subtype'].isin(common_subtypes),
        ['complaint_type', 'issue_subtype', 'resolution_days'],
    ]
    .groupby(['complaint_type', 'issue_subtype'])['resolution_days']
    .agg(
        complaints='size',
        median_days='median',
        p90_days=lambda s: s.quantile(0.9),
    )
    .reset_index()
    .sort_values(['complaint_type', 'median_days'], ascending=[True, False])
)
resolution_summary.groupby('complaint_type').head(5)

borough_mix = (
    nlp_df.loc[
        nlp_df['borough'].notna()
        & nlp_df['issue_subtype'].isin(common_subtypes)
        & nlp_df['complaint_type'].isin(['Illegal Parking', 'Noise - Residential', 'UNSANITARY CONDITION', 'Street Condition']),
        ['borough', 'issue_subtype'],
    ]
    .value_counts()
    .rename('complaints')
    .reset_index()
)
borough_mix.head(20)


,borough,issue_subtype,complaints
0,BRONX,music_party,184004
1,BROOKLYN,blocked_hydrant,91542
2,QUEENS,blocked_hydrant,74986
3,BROOKLYN,posted_sign_violation,69788
4,BROOKLYN,music_party,61865
5,BROOKLYN,banging_pounding,56534
6,BROOKLYN,blocked_sidewalk,51723
7,QUEENS,music_party,50969
8,QUEENS,posted_sign_violation,46397
9,BRONX,banging_pounding,45545


In [11]:
latest_2026_date = nlp_df.loc[nlp_df['created_year'].eq(2026), 'created_date'].max()
aligned_2025_cutoff = latest_2026_date - pd.DateOffset(years=1)

trend_frame = nlp_df.loc[
    nlp_df['issue_subtype'].isin(common_subtypes)
    & (
        ((nlp_df['created_date'] >= pd.Timestamp('2025-01-01')) & (nlp_df['created_date'] <= aligned_2025_cutoff))
        | ((nlp_df['created_date'] >= pd.Timestamp('2026-01-01')) & (nlp_df['created_date'] <= latest_2026_date))
    ),
    ['created_date', 'created_year', 'complaint_type', 'issue_subtype'],
].copy()
trend_frame['period'] = np.where(trend_frame['created_year'].eq(2026), '2026 YTD', '2025 aligned')

trend_summary = (
    trend_frame
    .groupby(['complaint_type', 'issue_subtype', 'period'])
    .size()
    .rename('complaints')
    .reset_index()
)
trend_summary['family_period_total'] = trend_summary.groupby(['complaint_type', 'period'])['complaints'].transform('sum')
trend_summary['share_within_family'] = trend_summary['complaints'] / trend_summary['family_period_total']

trend_shift = (
    trend_summary
    .pivot_table(index=['complaint_type', 'issue_subtype'], columns='period', values='share_within_family', fill_value=0)
    .reset_index()
)
trend_shift['share_change'] = trend_shift.get('2026 YTD', 0) - trend_shift.get('2025 aligned', 0)
trend_shift['share_change_pct_points'] = trend_shift['share_change'] * 100
trend_shift.sort_values('share_change', ascending=False).head(12)


period,complaint_type,issue_subtype,2025 aligned,2026 YTD,share_change,share_change_pct_points
23,Noise - Residential,banging_pounding,0.274769,0.452919,0.178151,17.815052
52,Street Condition,pothole,0.637858,0.759998,0.122141,12.214065
61,Water System,no_water,0.060721,0.145101,0.084381,8.438052
43,PLUMBING,no_water,0.206839,0.282175,0.075336,7.533562
0,Blocked Driveway,no_access,0.722620,0.779734,0.057114,5.711353
28,Noise - Street/Sidewalk,music_party,0.666989,0.710618,0.043629,4.362892
24,Noise - Residential,loud_talking,0.054963,0.083036,0.028073,2.807284
18,Illegal Parking,double_parked,0.078471,0.100003,0.021532,2.153239
31,PAINT/PLASTER,collapse_risk,0.173988,0.195434,0.021446,2.144568
12,Illegal Parking,blocked_crosswalk,0.054014,0.073277,0.019263,1.926316


In [12]:
overall_subtypes = (
    nlp_df.loc[nlp_df['issue_subtype'].isin(['not_modeled', 'other_unclassified']) == False]
    .groupby(['complaint_type', 'issue_subtype'])
    .size()
    .rename('complaints')
    .reset_index()
    .sort_values('complaints', ascending=False)
)

slowest_common = resolution_summary.loc[resolution_summary['complaints'] >= 1000].sort_values('median_days', ascending=False).iloc[0]
largest_shift = trend_shift.iloc[trend_shift['share_change'].abs().argmax()]
top_subtype = overall_subtypes.iloc[0]
largest_residual_cluster = residual_cluster_profiles.sort_values('cluster_rows', ascending=False).iloc[0] if not residual_cluster_profiles.empty else None
resolution_outcome_overall = (
    nlp_df.loc[
        nlp_df['complaint_type'].isin(TARGET_FAMILIES) & nlp_df['resolution_outcome_group'].ne('no_resolution_text'),
        'resolution_outcome_group',
    ]
    .value_counts()
)
matched_resolution_outcomes = resolution_outcome_overall.drop(labels=['other_resolution_language'], errors='ignore')
top_resolution_outcome = matched_resolution_outcomes.index[0] if not matched_resolution_outcomes.empty else resolution_outcome_overall.index[0]
top_resolution_outcome_count = int(matched_resolution_outcomes.iloc[0]) if not matched_resolution_outcomes.empty else int(resolution_outcome_overall.iloc[0])
mismatch_summary = nlp_df.loc[nlp_df['potential_label_mismatch_flag'], ['complaint_type', 'mismatch_suspected_family']].value_counts()
top_mismatch_note = (
    f"- The notebook flags {mismatch_rows:,} probable label-text mismatch row(s), led by {mismatch_summary.index[0][0]} text that reads closer to {mismatch_summary.index[0][1]}."
    if mismatch_rows else '- The notebook did not flag any probable label-text mismatches.'
)
residual_cluster_note = (
    f"- The biggest residual cluster candidate is {largest_residual_cluster['complaint_type']} with terms {largest_residual_cluster['cluster_terms']} ({int(largest_residual_cluster['cluster_rows']):,} complaints still outside the rule-based taxonomy)."
    if largest_residual_cluster is not None else '- Residual clustering did not surface any stable multi-row cluster candidates.'
)

findings = [
    f"- The modeled complaint families cover {modeled_rows / len(nlp_df):.1%} of all scoped complaints.",
    f"- Within modeled families, {assigned_rows / modeled_rows:.1%} of complaints receive a subtype label rather than falling into the residual bucket.",
    f"- The largest identified subtype is {top_subtype['complaint_type']} -> {top_subtype['issue_subtype']} with {int(top_subtype['complaints']):,} complaints.",
    top_mismatch_note,
    residual_cluster_note,
    f"- The most common modeled closure-language outcome is {top_resolution_outcome} with {top_resolution_outcome_count:,} complaints that include resolution text.",
    f"- Among common labeled subtypes, {slowest_common['complaint_type']} -> {slowest_common['issue_subtype']} has the slowest median valid resolution time at {slowest_common['median_days']:.2f} days.",
    f"- The biggest aligned 2026 YTD subtype-mix shift is {largest_shift['complaint_type']} -> {largest_shift['issue_subtype']} at {largest_shift['share_change_pct_points']:+.2f} percentage points versus the same 2025 window through {aligned_2025_cutoff.date()}.",
]

for finding in findings:
    print(finding)


- The modeled complaint families cover 59.6% of all scoped complaints.
- Within modeled families, 100.0% of complaints receive a subtype label rather than falling into the residual bucket.
- The largest identified subtype is Noise - Residential -> music_party with 342,520 complaints.
- The notebook flags 1 probable label-text mismatch row(s), led by Illegal Parking text that reads closer to Noise.
- Residual clustering did not surface any stable multi-row cluster candidates.
- The most common modeled closure-language outcome is no_evidence_observed with 560,211 complaints that include resolution text.
- Among common labeled subtypes, UNSANITARY CONDITION -> garbage_storage has the slowest median valid resolution time at 15.06 days.
- The biggest aligned 2026 YTD subtype-mix shift is Noise - Residential -> music_party at -21.58 percentage points versus the same 2025 window through 2025-04-10.


In [13]:
export_columns = [
    'unique_key',
    'created_date',
    'created_year',
    'agency',
    'borough',
    'community_board',
    'status',
    'complaint_type',
    'descriptor',
    'additional_details',
    'text_clean',
    'issue_family',
    'subtype_modeled_flag',
    'issue_subtype',
    'issue_subtype_confidence',
    'subtype_source',
    'subtype_match_text',
    'potential_label_mismatch_flag',
    'mismatch_suspected_family',
    'mismatch_match_text',
    'residual_cluster',
    'residual_cluster_label',
    'resolution_has_text',
    'resolution_outcome_group',
    'resolution_outcome_confidence',
    'resolution_text_length',
    'resolution_days',
    'resolved_with_valid_date',
]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
nlp_df[export_columns].to_parquet(OUTPUT_PATH, index=False)
print(f'Wrote subtype artifact to: {OUTPUT_PATH}')
print(f'Exported rows: {len(nlp_df):,}')


Wrote subtype artifact to: data/analytics/requests_2025_2026_issue_subtypes.parquet
Exported rows: 4,767,760
